In [ ]:
# Setup — run this cell first (hidden from slides)
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 13,
})

<div style="text-align:center; padding-top:60px;">

# Deep Learning Part 2
## From Pixels to Segmentation

**Convolutional Networks · Image Segmentation · Semantic Segmentation**

**MUSA 650 — Week 07**

---
⏱ 75 minutes &nbsp;|&nbsp; 4 parts &nbsp;|&nbsp; live code

</div>

## Today's Agenda

| # | Topic | Time |
|---|---|---|
| 1 | Why CNNs? The Limits of MLPs | ~10 min |
| 2 | Convolutional Networks (CNNs) | ~20 min |
| 3 | Image Segmentation — the problem | ~10 min |
| 4 | Semantic Segmentation — FCNs & U-Net | ~20 min |
| 5 | Lab walkthrough | ~15 min |

> **Goal**: understand why spatial structure matters, and how to train networks that predict *per-pixel* outputs.

---
# Part 1 — Why CNNs? The Limits of MLPs

## The MLP Problem: Treating Images as Flat Vectors

**MNIST MLP** (Part 1):
- 28 × 28 image → **flatten to 784** → fully connected layers
- **~407k parameters** for a tiny dataset
- **Final accuracy**: ~90%

### Why this is inefficient

```
Original structure:        After flattening:
  [pixel][pixel]            pixel₀
  [pixel][pixel]            pixel₁  ← spatial relationships LOST
                            pixel₂
                            ...
```

Three fundamental problems:

| Problem | Impact | Solution |
|---|---|---|
| **Spatial structure lost** | No notion of "nearby" pixels | Use local connections (convolution) |
| **Parameter explosion** | 28×28×1 = 784 inputs; even bigger for color | Parameter sharing (same filter everywhere) |
| **Translation sensitivity** | Pixel (5,5) and (6,5) treated as completely different | Hierarchical pooling |


In [ ]:
# Visualise the parameter explosion as image size grows

sizes = np.array([28, 64, 128, 256, 512])
mlp_params = sizes**2 * 128  # flatten -> fully connected to 128 neurons
cnn_params = np.array([3024, 16512, 65280, 261120, 1046528])  # Conv + pooling

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(sizes))
width = 0.35

bars1 = ax.bar(x - width/2, mlp_params/1e6, width, label='MLP (flatten + dense)', color='#e74c3c')
bars2 = ax.bar(x + width/2, cnn_params/1e6, width, label='CNN', color='#3498db')

ax.set_xlabel('Image Size (width × width)', fontsize=12)
ax.set_ylabel('Parameters (millions)', fontsize=12)
ax.set_title('Parameter Count: MLP vs CNN', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'{s}×{s}' for s in sizes])
ax.legend(fontsize=11)
ax.set_yscale('log')
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}M', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Beyond MNIST: Real-World Images

When images are bigger and more complex, MLPs fail catastrophically:

| Dataset | Size | # Classes | MLP | CNN |
|---|---|---|---|---|
| **MNIST** | 28×28 (B&W) | 10 | **90%** | **99%** |
| **CIFAR-10** | 32×32 (RGB) | 10 | 45% | **95%** |
| **ImageNet** | 224×224 (RGB) | 1000 | crashes (memOOM) | **92%** |
| **Medical imaging** | 512×512 (grayscale) | varies | unusable | **90%+** |

> The MLP parameters would exceed GPU memory before the network even sees the first image.

---
# Part 2 — Convolutional Networks (CNNs)

## The Convolution Operation

Instead of fully connected layers, use **filters** that slide across the image:

```
Input image (28×28)              Filter (3×3)
                                  ┌─────────┐
  ┌──────────────────────────┐   │-0.1  0.2│
  │█████████████████████████░│   │ 0.5  0.3│
  │█████████████████████████░│   │ 0.0 -0.2│
  │█████████████████████████░│   └─────────┘
  │█████████████████████████░│
  └──────────────────────────┘
   ↓ apply filter at every (x,y)
Output (26×26)  ← smaller due to padding
  ┌──────────────────────────┐
  │0.5 0.3 0.1 -0.2 ... ░░░░░│
  │0.2 0.6 0.4  0.1 ... ░░░░░│
  │0.1 0.3 0.7  0.2 ... ░░░░░│
  └──────────────────────────┘
```

### Key ideas:
1. **Local connectivity**: each output depends on a small neighborhood (e.g. 3×3)
2. **Weight sharing**: same filter weights at every spatial location
3. **Spatial hierarchy**: deeper layers see larger receptive fields

In [ ]:
# Visualise a simple convolution operation

# Create a simple image
image = np.array([
    [1, 2, 3, 0, 0],
    [0, 1, 2, 0, 0],
    [0, 0, 1, 2, 0],
    [0, 0, 0, 1, 2],
    [0, 0, 0, 0, 1]
], dtype=float)

# Create a filter (edge detector)
kernel = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=float)

# Manual convolution (for visualization)
output = np.zeros((3, 3))
for i in range(3):
    for j in range(3):
        patch = image[i:i+3, j:j+3]
        output[i, j] = np.sum(patch * kernel)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

axes[0].imshow(image, cmap='gray', vmin=-3, vmax=3)
axes[0].set_title('Input Image (5×5)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Rows')
for i in range(image.shape[0]):
    for j in range(image.shape[1]):
        axes[0].text(j, i, f'{image[i,j]:.0f}', ha='center', va='center', 
                     color='white' if image[i,j] > 0.5 else 'black', fontsize=10)

axes[1].imshow(kernel, cmap='RdBu_r', vmin=-2, vmax=2)
axes[1].set_title('Filter: Sobel (3×3)\n(edge detection)', fontsize=12, fontweight='bold')
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{kernel[i,j]:.1f}', ha='center', va='center',
                     color='white' if abs(kernel[i,j]) > 0.5 else 'black', fontsize=10)

axes[2].imshow(output, cmap='RdBu_r', vmin=-10, vmax=10)
axes[2].set_title('Output (3×3)\nconvolution', fontsize=12, fontweight='bold')
for i in range(3):
    for j in range(3):
        axes[2].text(j, i, f'{output[i,j]:.0f}', ha='center', va='center',
                     color='white' if abs(output[i,j]) > 3 else 'black', fontsize=10)

plt.tight_layout()
plt.show()

print(f"Input shape: {image.shape}")
print(f"Filter shape: {kernel.shape}")
print(f"Output shape: {output.shape}")
print(f"\nFilter learns to detect **vertical edges** (Sobel operator)")

## Output Size Calculation

Given:
- Input: $H \times W$
- Filter: $F \times F$
- Padding: $P$
- Stride: $S$

Output:
$$H_{\text{out}} = \left\lfloor \frac{H + 2P - F}{S} \right\rfloor + 1$$

### Examples (stride=1, no padding):

| Input | Filter | Output | Formula |
|---|---|---|---|
| 28×28 | 3×3 | 26×26 | (28 - 3 + 1) = 26 |
| 28×28 | 5×5 | 24×24 | (28 - 5 + 1) = 24 |
| 224×224 | 3×3 | 222×222 | (224 - 3 + 1) = 222 |

### With padding=1 (keep size):
| Input | Filter | Output |
|---|---|---|
| 28×28 | 3×3 | 28×28 ← useful! |
| 224×224 | 3×3 | 224×224 |


## Multiple Filters → Feature Maps

In practice, we use **many** filters (e.g., 32, 64, 128).
Each learns a different feature.

```
Input (28×28×1)           Conv2D(32 filters, 3×3)
                           ↓
  Single image           32 feature maps (26×26 each)
                           ↓
  [pixel values]         [edge detectors]
                         [corner detectors]
                         [texture detectors]
                         [blob detectors]
                         ...
```

### Parameter count for Conv2D:
$$N_{\text{params}} = (F \times F \times C_{\text{in}} + 1) \times C_{\text{out}}$$

**Example**: Conv2D(32 filters, 3×3, input=1 channel)
- Per filter: $3 \times 3 \times 1 + 1 = 10$ parameters
- Total: $10 \times 32 = 320$ parameters

**Compare to fully connected** (28×28 → 32 neurons): $784 \times 32 = 25,088$ parameters! ✗

## Pooling Layers

Reduce spatial dimensions while preserving important information:

```
Max Pooling (2×2, stride=2):      Average Pooling (2×2, stride=2):

  4  3    5 ←─ max                  4  3    3.75 ← average
  2  5    ┘                         2  5    ┘

  1  2    3                         1  2    2
  0  3    ┘                         0  3    ┘
```

| Layer | Input | Output | Purpose |
|---|---|---|---|
| **Conv2D(32, 3×3)** | 28×28 | 26×26 | Detect local patterns |
| **MaxPooling2D(2×2)** | 26×26 | 13×13 | Coarsen, reduce memory |
| **Conv2D(64, 3×3)** | 13×13 | 11×11 | Detect larger patterns |
| **MaxPooling2D(2×2)** | 11×11 | 5×5 | Coarsen further |
| **Flatten** | 5×5×64 | 1600 | Reshape for dense layers |
| **Dense(128)** | 1600 | 128 | Classification |

> **Max pooling is more common** — it preserves the "strongest activation" in each region.

In [ ]:
# Visualise max pooling

feature_map = np.array([
    [0.2, 0.5, 0.1, 0.3],
    [0.1, 0.8, 0.4, 0.2],
    [0.3, 0.2, 0.6, 0.1],
    [0.4, 0.1, 0.7, 0.9]
])

# Max pool (2×2)
pooled = np.array([
    [0.8, 0.4],
    [0.4, 0.9]
])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Input
im1 = ax1.imshow(feature_map, cmap='viridis', vmin=0, vmax=1)
ax1.set_title('Feature Map (4×4)', fontsize=12, fontweight='bold')
for i in range(4):
    for j in range(4):
        ax1.text(j, i, f'{feature_map[i,j]:.1f}', ha='center', va='center',
                 color='white', fontsize=10, fontweight='bold')

# Draw pooling windows
rect1 = mpatches.Rectangle((0, 0), 2, 2, linewidth=3, edgecolor='red', facecolor='none')
rect2 = mpatches.Rectangle((2, 0), 2, 2, linewidth=3, edgecolor='red', facecolor='none')
rect3 = mpatches.Rectangle((0, 2), 2, 2, linewidth=3, edgecolor='red', facecolor='none')
rect4 = mpatches.Rectangle((2, 2), 2, 2, linewidth=3, edgecolor='red', facecolor='none')
ax1.add_patch(rect1)
ax1.add_patch(rect2)
ax1.add_patch(rect3)
ax1.add_patch(rect4)

# Output
im2 = ax2.imshow(pooled, cmap='viridis', vmin=0, vmax=1)
ax2.set_title('After Max Pooling (2×2)', fontsize=12, fontweight='bold')
ax2.set_xticks([0, 1])
ax2.set_yticks([0, 1])
for i in range(2):
    for j in range(2):
        ax2.text(j, i, f'{pooled[i,j]:.1f}', ha='center', va='center',
                 color='white', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## A Complete CNN for MNIST

```
Input (28×28×1)
  ↓
Conv2D(32, kernel_size=(3,3), activation='relu')
  ↓  output: 26×26×32
Conv2D(64, (3,3), activation='relu')
  ↓  output: 24×24×64
MaxPooling2D(pool_size=(2,2))
  ↓  output: 12×12×64
Dropout(0.25)
  ↓
Flatten() → 9,216 features
  ↓
Dense(128, activation='relu')
  ↓
Dropout(0.5)
  ↓
Dense(10, activation='softmax')
  ↓
Output probabilities (10 classes)
```

### Comparison: MLP vs CNN on MNIST

| Metric | MLP | CNN |
|---|---|---|
| Parameters | 407,050 | 1,199,882 |
| Test Accuracy | 90% | 99%+ |
| Training time | ~10 sec/epoch | ~150 sec/epoch |
| Memory | Low | Medium |

> More parameters but **far better** accuracy. The spatial structure matters!

---
# Part 3 — Image Segmentation (The Problem)

## Classification vs Segmentation

### Image Classification (what we've done so far)
```
Input image          Network          Output
                                       "dog" (probability)
[28×28×3]  ──────► [hidden layers] ──► [0.92]
                                       (single class label)
```

**Question**: *What is in the image?*

### Image Segmentation (pixel-level prediction)
```
Input image          Network          Output
                                       Pixel-wise labels
[H×W×3]    ──────► [hidden layers] ──► [H×W×K]
                                       (K = number of classes)
                                       Each pixel gets a label
```

**Question**: *Which pixels belong to which object?*

### Example: Autonomous Driving
```
Input:                            Segmentation output:
┌─────────────────────┐           ┌─────────────────────┐
│ 🚗 🚙 🚗             │           │ RRR GGG GGG GGG GGG │
│ 🛣️  🛣️  🛣️  🛣️  🛣️        │           │ RRR GGG GGG GGG GGG │
│ 👤 👤 🛣️  🌳 🌳     │           │ BBB GGG GGG YYY YYY │
│ 👤 👤 🛣️  🌳 🌳     │           │ BBB GGG GGG YYY YYY │
└─────────────────────┘           └─────────────────────┘

Classes: R=vehicle G=road B=pedestrian Y=tree
```

## Segmentation Tasks

### 1. Semantic Segmentation
**Assign a class to every pixel** (no distinction between instances)

```
Input:                    Output:
  🐕  🐕                    dog  dog
grass grass                grass grass

Both dogs → same color ("dog" class)
```

**Example applications**:
- Medical imaging (tumor vs normal tissue)
- Satellite imagery (water, forest, urban)
- Autonomous driving

---

### 2. Instance Segmentation
**Distinguish individual objects of the same class**

```
Input:                    Output:
  🐕  🐕                    dog₁ dog₂
grass grass                grass grass

First dog → red, second dog → blue
```

**Example applications**:
- Cell counting in microscopy
- Crowd analysis
- Object detection with masks

---

### 3. Panoptic Segmentation
**Combine semantic + instance segmentation** (stuff + things)

---

> **In this course, we focus on SEMANTIC SEGMENTATION**

In [ ]:
# Create a visual comparison of segmentation tasks

fig, axes = plt.subplots(2, 3, figsize=(12, 6))

# Simple synthetic image
img = np.zeros((100, 100, 3), dtype=np.uint8)
img[20:50, 20:45] = [255, 0, 0]      # red object 1
img[20:50, 55:80] = [255, 0, 0]      # red object 2
img[60:90, 30:70] = [0, 255, 0]      # green object

# Labels for semantic segmentation (0=background, 1=red, 2=green)
semantic = np.zeros((100, 100), dtype=np.uint8)
semantic[20:50, 20:45] = 1
semantic[20:50, 55:80] = 1
semantic[60:90, 30:70] = 2

# Labels for instance segmentation (0=background, 1=red obj1, 2=red obj2, 3=green)
instance = np.zeros((100, 100), dtype=np.uint8)
instance[20:50, 20:45] = 1
instance[20:50, 55:80] = 2
instance[60:90, 30:70] = 3

# Plot
axes[0, 0].imshow(img)
axes[0, 0].set_title('Input Image', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(semantic, cmap='tab10')
axes[0, 1].set_title('Semantic Segmentation\n(class labels)', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].imshow(instance, cmap='tab10')
axes[0, 2].set_title('Instance Segmentation\n(instance IDs)', fontsize=12, fontweight='bold')
axes[0, 2].axis('off')

# Add legend row
axes[1, 0].axis('off')
axes[1, 1].axis('off')
axes[1, 2].axis('off')

legend_text = '''Semantic: treats all red objects as one class
Instance: distinguishes object 1 vs object 2'''
axes[1, 1].text(0.5, 0.5, legend_text, ha='center', va='center',
                fontsize=11, transform=axes[1, 1].transAxes,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

---
# Part 4 — Semantic Segmentation Architectures

## The Problem with Naive CNN

If we just use a classification CNN with pooling, we lose spatial information:

```
Input (256×256)           After pooling (4×4)
  High resolution           Very coarse
    image                   ↓
                    Can't reconstruct original pixels!
                    Upsampling is needed.
```

### Solution: Fully Convolutional Networks (FCNs)

Replace fully connected layers with **deconvolution** (transpose convolution):

```
Input (H×W×C)                Output (H×W×K)
  ↓                             ↑
Conv + Pool (shrink)        Deconv + Upsample (expand)
  ↓                             ↑
Bottleneck (smallest)       Reconstructed segmentation
```

**FCN-32s** (upsampling by 32×): Fast, coarse
**FCN-16s** (upsampling by 16×): Adds skip connections
**FCN-8s** (upsampling by 8×): More details from early layers

## U-Net Architecture

**U-Net** is a encoder-decoder network with **skip connections**:

```
Encoder (shrinking)              Decoder (growing)     Output

Input (572×572×1)                                   (572×572×2)
  │                                                     ↑
  ├─ Conv 64   ─┐                              ┌────UpConv 
  │              │                              │
  ├─ MaxPool    │  └──────────────────────────┼───────────
  │              ↓
  ├─ Conv 128   │         skip connection      │
  │              │                              │
  ├─ MaxPool    │  └───────────────────────────┼─────────
  │              ↓
  ├─ Conv 256   │     bottleneck                │
  │              │                              │
  └─ MaxPool    │  └─────┬──────────────────────┼──────
                 ↓        │                      │
              Conv 512    │                      │
                          ↓                      ↓
                      UpConv(concat)  →  Output conv  →  [output]
```

### Key innovations:
1. **Skip connections**: concatenate encoder features with decoder (↗ details)
2. **Encoder-decoder**: contracting path + expanding path
3. **Fewer parameters**: ~1.9M vs millions for FCN
4. **Data efficient**: works well even with limited training data

> U-Net won first place in the 2015 ISBI Cell Tracking Challenge.

In [ ]:
# Visualise U-Net architecture diagram

fig, ax = plt.subplots(figsize=(14, 6))

# Define layer sizes
layers = [
    ('Input\n572×572×1', 0, 5, 'lightblue'),
    ('Conv 64\n570×570×64', 1, 5, 'skyblue'),
    ('MaxPool\n285×285×64', 2, 5, 'steelblue'),
    ('Conv 128\n283×283×128', 3, 5, 'lightgreen'),
    ('MaxPool\n141×141×128', 4, 5, 'forestgreen'),
    ('Conv 256\n139×139×256', 5, 5, 'lightyellow'),
    ('MaxPool\n69×69×256', 6, 5, 'gold'),
    ('Conv 512\n67×67×512', 7, 5, 'lightcoral'),
    ('Bottleneck', 8, 2.5, 'darkred'),
    ('UpConv\n134×134×256', 9, 5, 'salmon'),
    ('UpConv\n268×268×128', 10, 5, 'lightgreen'),
    ('UpConv\n536×536×64', 11, 5, 'lightblue'),
    ('Output\n572×572×2', 12, 5, 'navy'),
]

# Draw boxes
for label, x, h, color in layers:
    if x < 8:
        ax.add_patch(mpatches.Rectangle((x-0.3, 2.5-h/2), 0.6, h, 
                                        facecolor=color, edgecolor='black', linewidth=1.5))
    elif x == 8:
        ax.add_patch(mpatches.Rectangle((x-0.4, 2.5-h/2), 0.8, h,
                                        facecolor=color, edgecolor='black', linewidth=2))
    else:
        ax.add_patch(mpatches.Rectangle((x-0.3, 2.5-h/2), 0.6, h,
                                        facecolor=color, edgecolor='black', linewidth=1.5))
    ax.text(x, 2.5, label, ha='center', va='center', fontsize=8, fontweight='bold')

# Draw arrows for encoding path
for i in range(7):
    ax.arrow(layers[i][1]+0.35, 2.5, 0.3, 0, head_width=0.2, head_length=0.1, fc='black', ec='black')

# Draw arrow to bottleneck
ax.arrow(7.35, 2.5, 0.25, 0, head_width=0.2, head_length=0.1, fc='black', ec='black')

# Draw arrows for decoding path
for i in range(9, 13):
    ax.arrow(layers[i][1]-0.35, 2.5, -0.3, 0, head_width=0.2, head_length=0.1, fc='black', ec='black')

# Draw skip connections (curved arrows)
skip_pairs = [(2, 10), (4, 9), (6, 11)]  # connect pooled layers to upsampled layers
for enc_idx, dec_idx in skip_pairs:
    enc_x, dec_x = layers[enc_idx][1], layers[dec_idx][1]
    from matplotlib.patches import FancyArrowPatch
    arrow = FancyArrowPatch((enc_x, 4), (dec_x, 1), 
                           connectionstyle='arc3,rad=0.5', 
                           arrowstyle='->', mutation_scale=15,
                           color='red', linewidth=1.5, linestyle='--')
    ax.add_patch(arrow)

ax.text(6.5, 0.5, 'Skip connections concatenate features', fontsize=10, color='red', fontweight='bold')

ax.set_xlim(-1, 13.5)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_aspect('equal')
ax.set_title('U-Net Architecture (simplified)', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

## Loss Function for Segmentation

### Per-pixel Cross-Entropy
Apply cross-entropy **independently to each pixel**:

$$\mathcal{L} = -\frac{1}{N \times H \times W} \sum_{n,h,w} \sum_{k} y_{n,h,w,k} \log(\hat{y}_{n,h,w,k})$$

Where:
- $N$ = batch size
- $H, W$ = height, width
- $K$ = number of classes
- $y_{n,h,w,k}$ = true label at pixel $(h,w)$ in image $n$
- $\hat{y}_{n,h,w,k}$ = predicted probability

### Class Imbalance Problem
If one class dominates (e.g., 95% background), the network ignores rare classes.

**Solutions**:
1. **Weighted cross-entropy**: upweight rare classes
2. **Dice loss**: focus on overlap, less sensitive to class imbalance
3. **Focal loss**: down-weight easy samples

**Weighted Cross-Entropy**:
$$\mathcal{L} = -\sum_k w_k \, y_k \log(\hat{y}_k)$$

where $w_k$ is higher for rarer classes.

## Evaluation Metrics for Segmentation

### Pixel Accuracy
$$\text{Pixel Acc} = \frac{\text{# correct pixels}}{\text{# total pixels}}$$
**Problem**: insensitive to class imbalance (95% accuracy if predicting "background" everywhere)

---

### IoU (Intersection over Union) — Primary metric ⭐
$$\text{IoU}_k = \frac{|\text{Predicted}_k \cap \text{Ground Truth}_k|}{|\text{Predicted}_k \cup \text{Ground Truth}_k|}$$

```
Predicted:    GT:           IoU = overlap / union
  ███             ██          |
  ███       +     ██       =   ██   / ███  =  2/4 = 0.5
   ██             ███         ███      ███
                                       ███
```

- **IoU = 0**: no overlap
- **IoU = 1**: perfect prediction
- More robust to class imbalance

---

### Mean IoU (mIoU)
$$\text{mIoU} = \frac{1}{K} \sum_{k=1}^{K} \text{IoU}_k$$

**Standard metric in segmentation benchmarks** (Pascal VOC, Cityscapes, etc.)

In [ ]:
# Visualise IoU calculation

gt = np.zeros((6, 6), dtype=int)
gt[1:4, 1:5] = 1  # ground truth region

pred = np.zeros((6, 6), dtype=int)
pred[2:5, 2:5] = 1  # predicted region (partially overlaps)

overlap = np.logical_and(gt, pred).astype(int)
union = np.logical_or(gt, pred).astype(int)

iou = np.sum(overlap) / np.sum(union)

fig, axes = plt.subplots(2, 2, figsize=(8, 8))

# Ground truth
axes[0, 0].imshow(gt, cmap='Reds', vmin=0, vmax=2)
axes[0, 0].set_title('Ground Truth (label=1)', fontsize=11, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Prediction
axes[0, 1].imshow(pred, cmap='Blues', vmin=0, vmax=2)
axes[0, 1].set_title('Prediction (label=1)', fontsize=11, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Overlap
axes[1, 0].imshow(overlap, cmap='Greens', vmin=0, vmax=2)
axes[1, 0].set_title(f'Overlap (intersection)\n{np.sum(overlap)} pixels', fontsize=11, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Union
axes[1, 1].imshow(union, cmap='Purples', vmin=0, vmax=2)
axes[1, 1].set_title(f'Union\n{np.sum(union)} pixels', fontsize=11, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

fig.text(0.5, 0.02, f'IoU = {np.sum(overlap)} / {np.sum(union)} = {iou:.2%}', 
         ha='center', fontsize=14, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

---
# Part 5 — Lab Walkthrough

## Lab Structure: From MLP → CNN → Segmentation

### Week 6 Part 1 (Completed) ✓
- **Goal**: Build an MLP on MNIST
- **Dataset**: MNIST (28×28, 10 classes)
- **Architecture**: 784 → 512 → 10
- **Result**: ~90% accuracy

### Week 6 Part 2 (Current)
- **Goal**: Build a CNN on MNIST
- **Dataset**: MNIST (28×28, 10 classes)
- **Architecture**: Conv32 → MaxPool → Conv64 → Flatten → Dense128 → Dense10
- **Expected result**: ~99% accuracy
- **Task**: Fill in the blanks in `DLBasics_SimpleCNN.ipynb`

### Week 7 Part 3 (Advanced)
- **Goal**: Semantic segmentation on custom dataset
- **Architecture**: U-Net or FCN
- **Task**: Build a network that predicts per-pixel labels

## CNN Lab at a Glance

Open **`DLBasics_SimpleCNN.ipynb`** — similar structure to Part 1:

| Section | Task | Concept |
|---|---|---|
| 1 | Imports | Same as before |
| 2 | Load MNIST | `mnist.load_data()` |
| 3 | Preprocess | **Keep 2D shape** (no flatten!) |
| 4 | One-hot encode | Same as before |
| 5 | Build CNN | `Conv2D` + `MaxPooling2D` layers |
| 6 | Compile | `loss`, `optimizer`, `metrics` |
| 7 | Train | `model.fit` (more epochs needed) |
| 8 | Evaluate | `model.evaluate` |

### Key Differences from MLP Lab

**Preprocessing (Step 3)**:
```python
# MLP: flatten
x_train = x_train.reshape(60000, 784)

# CNN: keep 2D structure
x_train = x_train.reshape(60000, 28, 28, 1)  # 4D: (batch, height, width, channels)
```

**Model Building (Step 5)**:
```python
# MLP: Dense layers
model.add(Dense(512, activation='relu', input_shape=(784,)))

# CNN: Convolutions + Pooling
model.add(Conv2D(32, kernel_size=(3,3), activation='relu', input_shape=(28,28,1)))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Conv2D(64, (3,3), activation='relu'))
# ... more layers
model.add(Flatten())
model.add(Dense(128, activation='relu'))
```

## Common CNN Architecture Patterns

### Pattern 1: Simple Progression
```
Conv(32) → MaxPool → Conv(64) → MaxPool → Flatten → Dense(128) → Dense(10)
```
Simple, fast, ~99% on MNIST

---

### Pattern 2: Deeper (ResNet-like)
```
Conv(32) → Conv(32)                     (residual connection)
   ↓              ↓
   └─ Add ──→ MaxPool → Conv(64) → Conv(64) → MaxPool → ...
```
Harder to train, but can reach higher accuracy with careful tuning

---

### Pattern 3: Multi-Scale (Inception-like)
```
     Conv(1×1)     Conv(3×3)     Conv(5×5)
        ↓             ↓             ↓
    [parallel processing]
        ↓             ↓             ↓
     ──────────────────────────────── Concatenate
```
Captures multiple scales simultaneously

---

> **For this lab, stick with Pattern 1 — it's simple and effective.**

## Optimization Tips for CNNs

### 1. Learning Rate
- Too high → training oscillates wildly
- Too low → training is painfully slow
- Start with `0.001` (default for Adam), adjust by factor of 10

### 2. Dropout Location
```python
# After pooling (already downsampled)
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))  # ← here

# Before final output
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))   # ← here
```

### 3. Batch Size vs Gradient Noise
- Large batch (e.g., 128) → stable but fewer updates per epoch
- Small batch (e.g., 32) → noisier but more frequent updates
- Usually **32–128** is sweet spot for modern hardware

### 4. Data Augmentation (for overfitting)
```python
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)

model.fit(datagen.flow(x_train, y_train, batch_size=32), ...)
```

### 5. Early Stopping (when to stop training)
```python
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.fit(x_train, y_train, callbacks=[early_stop], ...)
```

## Exercises After Completing the Lab

### Beginner
1. **Visualise filters**: Extract and display what each Conv2D filter learned
   ```python
   weights = model.layers[0].get_weights()[0]
   plt.imshow(weights[:, :, 0, 5])  # 5th filter of layer 0
   ```

2. **Visualise activations**: See what the hidden layers output for a sample
   ```python
   from tensorflow.keras.models import Model
   intermediate_layer = Model(inputs=model.input, outputs=model.layers[2].output)
   ```

### Intermediate
3. **Add batch normalization**: Stabilises training, allows higher learning rates
   ```python
   from keras.layers import BatchNormalization
   model.add(Conv2D(32, (3,3), activation='relu'))
   model.add(BatchNormalization())
   ```

4. **Add more convolutional blocks**: Deeper = more feature extraction
   ```python
   model.add(Conv2D(128, (3,3), activation='relu'))
   model.add(MaxPooling2D(pool_size=(2,2)))
   ```

### Advanced
5. **Transfer learning**: Use a pre-trained network (VGG16, ResNet50)
   ```python
   from tensorflow.keras.applications import ResNet50
   base_model = ResNet50(weights='imagenet', include_top=False)
   # Fine-tune on MNIST
   ```

6. **Build a U-Net for segmentation** (Week 7 prep)
   - Skip connections from encoder to decoder
   - Output shape matches input shape

## Semantic Segmentation Lab Preview (Week 7)

You'll move from **classification** (one label per image) to **segmentation** (one label per pixel):

### Data Format Change
```python
# Classification
x_train shape: (60000, 28, 28, 1)
y_train shape: (60000, 10)  ← one label per image

# Segmentation
x_train shape: (1000, 256, 256, 3)  ← high-res images
y_train shape: (1000, 256, 256, 5)  ← per-pixel labels (5 classes)
```

### Architecture Change
```python
# Classification CNN (ends with Dense)
model = Sequential([
    Conv2D(32, ...), MaxPooling2D(), 
    Conv2D(64, ...), MaxPooling2D(),
    Flatten(), Dense(10, activation='softmax')
])

# Segmentation U-Net (encoder + decoder + skip connections)
# Output shape = input shape (e.g., 256×256)
```

### Loss Change
```python
# Classification
loss='categorical_crossentropy'

# Segmentation (same idea, but per pixel)
loss='categorical_crossentropy'  # applied to each pixel
# OR
loss=dice_loss  # emphasises overlap (Dice coefficient)
```

## Key Takeaways

1. **CNNs preserve spatial structure** → dramatically better than MLPs on images
   - Convolution: local connectivity + weight sharing
   - Pooling: coarsen representation, build hierarchy

2. **Semantic segmentation = per-pixel classification**
   - Output shape = input shape (e.g., 256×256×K)
   - Requires encoder-decoder architecture (U-Net, FCN)

3. **Skip connections are crucial**
   - Encoder features concatenated with decoder
   - Preserves fine spatial details

4. **Evaluation metrics differ**
   - Classification: accuracy, precision, recall
   - Segmentation: IoU (Intersection over Union), mIoU

5. **Real-world segmentation is harder**
   - Class imbalance (rare objects)
   - Need large, diverse training datasets
   - Transfer learning from pretrained models helps

## Resources

**Papers**
- [Convolutional Networks (LeCun et al., 1998)](http://yann.lecun.com/exdb/publis/pdf/lecun-98.pdf)
- [U-Net (Ronneberger et al., 2015)](https://arxiv.org/abs/1505.04597)
- [FCN (Long et al., 2015)](https://arxiv.org/abs/1411.4038)

**Visualisations**
- [Conv Playground](https://poloclub.github.io/cnn-explainer/) — interactive CNN builder
- [Feature Visualisation (OpenAI)](https://openai.com/research/features-neurons-in-neural-networks/)

**Tutorials**
- Keras docs: [CNNs](https://keras.io/examples/vision/mnist_convnet/)
- CS231n: [CNNs](https://cs231n.github.io/convolutional-networks/)
- PyTorch: [U-Net Implementation](https://github.com/milesial/Pytorch-UNet)

**Datasets**
- **PASCAL VOC** (semantic segmentation)
- **Cityscapes** (autonomous driving)
- **Medical Imaging**: BraTS (brain tumors), ISBI (cells)
- **Satellite**: Sentinel, Landsat

<div style="text-align:center; padding-top:60px;">

# Questions?

---

**Next steps:**
1. Complete CNN Lab (Part 2)
2. Experiment with the suggested exercises
3. Prepare for semantic segmentation (Week 7)

*Solutions will be posted after the lab deadline*

</div>